# A2 - Predict next-hour road traffic
**Question:** Can recent traffic counts and the next hour's clock time predict the number of vehicles in the next hour better than simply repeating the latest count?

**Data:** UCI Metro Interstate Traffic Volume (I-94 westbound, Minnesota, 2012-2018), https://archive.ics.uci.edu/dataset/492/metro+interstate+traffic+volume . Credit: John Hogue / UCI Machine Learning Repository (CC BY 4.0). One location only; this is a study, not a citywide traffic service.

**How to read this notebook:** Each numbered section states what it does, why it matters, and a small amount of code. Run from top to bottom in a fresh Colab CPU runtime. No Google Drive mount or private file is required. The input is three recent hourly counts plus the upcoming hour and weekday. The output is a predicted vehicle count. No same-hour or future weather information is used.
**Why this matters:** The U.S. Federal Highway Administration describes short-term forecasts as one input to active traffic management. This notebook only predicts the demand count at one sensor; it cannot prescribe a traffic-control action or estimate travel speed. A last-hour rule misses systematic changes around rush hours, motivating a model that can learn time and recent-count patterns. Source: https://ops.fhwa.dot.gov/publications/fhwahop19035/ch4.htm


**Practical scope and conditions for learning:** The immediate objective is an accurate next-hour count at this sensor. Such a forecast could inform planning, but this experiment does not demonstrate an improvement in traffic-control decisions. Learning is feasible because many historical input–target examples are available, inputs are known before the target hour, and recurring daily patterns may persist. A historical average captures the clock pattern but cannot adjust to today's recent counts; repeating the last count adapts to current volume but ignores the upcoming hour. The tree combines both kinds of information. Transfer to another station or a later period requires fresh validation because these relationships may change.


## 0. Environment setup
A fresh Colab runtime usually has the scientific Python packages. This cell installs only missing dependencies and prints versions so the run can be reproduced.


In [ ]:
import importlib.util, subprocess, sys
packages = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib', 'sklearn':'scikit-learn'}
missing = [pip_name for module,pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import numpy, pandas, matplotlib, sklearn
print('Python:', sys.version.split()[0], 'numpy:',numpy.__version__,
      'pandas:',pandas.__version__, 'matplotlib:',matplotlib.__version__,
      'scikit-learn:',sklearn.__version__)


## 1. Load the public measurements
One raw row contains a timestamp, measured traffic volume, and weather information for that hour. Our prediction will use the traffic counts and the clock, because the actual weather *next* hour would not yet be known. The dataset is downloaded inside the notebook so anyone can reproduce the work.

In [ ]:
import io, gzip, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

URL = 'https://archive.ics.uci.edu/static/public/492/metro+interstate+traffic+volume.zip'
with urllib.request.urlopen(URL, timeout=60) as response:
    downloaded = response.read()
with zipfile.ZipFile(io.BytesIO(downloaded)) as archive:
    raw_bytes = gzip.decompress(archive.read('Metro_Interstate_Traffic_Volume.csv.gz'))
raw = pd.read_csv(io.BytesIO(raw_bytes), parse_dates=['date_time'])
print('Raw rows:', len(raw), 'columns:', list(raw.columns))
print(raw[['date_time','traffic_volume']].head().to_string(index=False))

## 2. Make one observation per hour
The source sometimes lists several weather conditions at the same timestamp. Their traffic count is identical. We check this, then keep one count per timestamp. Otherwise the model would see the same answer multiple times. We deliberately avoid filling gaps: if an hour is missing, we cannot pretend to know its count.

In [ ]:
distinct_counts = raw.groupby('date_time')['traffic_volume'].nunique()
assert distinct_counts.max() == 1, 'Duplicate hours disagree on the count; inspect before continuing.'
hourly = (raw[['date_time','traffic_volume']]
          .drop_duplicates('date_time')
          .sort_values('date_time')
          .reset_index(drop=True))
print('Unique measured hours:', len(hourly))
print('Duplicate raw rows removed:', len(raw)-len(hourly))

## 3. Build examples without seeing the future
At the end of hour `t`, the counts for hours `t`, `t-1`, and `t-2` are known. We predict hour `t+1`. That hour's clock time and weekday are also known in advance. We keep only examples with four consecutive measured hours; a gap is not an ordinary next-hour example.

Example: if the three latest hours had 4,100, 4,500, and 4,700 vehicles, the model uses those three numbers and the upcoming hour to estimate the next count. `next_volume` is the answer used *during training*; it is never an input at deployment.

**Exact interfaces:** One training row has five numeric input columns: `latest_volume` (hour t), `previous_1` (t−1), `previous_2` (t−2), `next_hour` (integer 0–23), and `next_weekday` (integer 0–6, Monday=0). Count inputs are finite, nonnegative vehicles per measured hour from the same sensor and consecutive hours. The supervised target is `next_volume`, a nonnegative count for t+1. Deployment takes the same five inputs and returns one nonnegative floating-point count estimate; a fractional prediction is an expected count, not a fractional vehicle observation. The caller must ensure valid inputs; the example function does not implement full input validation. This system assumes that the completed hour's measurement is available before issuing the forecast.


In [ ]:
frame = hourly.copy()
frame['previous_1'] = frame.traffic_volume.shift(1)
frame['previous_2'] = frame.traffic_volume.shift(2)
frame['next_volume'] = frame.traffic_volume.shift(-1)
frame['next_time'] = frame.date_time + pd.Timedelta(hours=1)
frame['next_hour'] = frame.next_time.dt.hour
frame['next_weekday'] = frame.next_time.dt.dayofweek  # Monday=0, Sunday=6
valid = ((frame.date_time - frame.date_time.shift(1) == pd.Timedelta(hours=1)) &
         (frame.date_time.shift(1) - frame.date_time.shift(2) == pd.Timedelta(hours=1)) &
         (frame.date_time.shift(-1) - frame.date_time == pd.Timedelta(hours=1)))
data = frame.loc[valid].copy()
data = data.rename(columns={'traffic_volume':'latest_volume'})
features = ['latest_volume','previous_1','previous_2','next_hour','next_weekday']
assert data[features+['next_volume']].notna().all().all()
print('Valid next-hour examples:',len(data))
print(data[['date_time']+features+['next_volume']].head().to_string(index=False))

## 4. Test on later years
A random row split would mix earlier and later measurements. Instead we train on target hours before 2017, choose model depth on 2017, then evaluate once on 2018. This simulates using past records to predict future traffic. The model never receives `next_volume` as an input.

**Why this is a valid one-step test:** A forecast early in 2018 may use observed counts from late 2017: those counts were already available when the forecast would be issued. This is not target leakage. Each test forecast receives the actual preceding observations, so the experiment evaluates rolling one-hour-ahead forecasts, not a recursive forecast made for all of 2018 at once. Missing sequences are excluded rather than imputed; the scores therefore apply to hours with usable recent measurements.


In [ ]:
train = data[data.next_time < '2017-01-01'].copy()
validation = data[(data.next_time >= '2017-01-01') & (data.next_time < '2018-01-01')].copy()
test = data[data.next_time >= '2018-01-01'].copy()
for name, part in [('train',train),('validation',validation),('test',test)]:
    print(name, 'examples:',len(part), 'from',part.next_time.min(),'to',part.next_time.max())
    assert len(part) > 1000
assert train.next_time.max() < validation.next_time.min() < test.next_time.min()

## 5. First benchmark: repeat the last hour
Two simple forecasts set a fair standard. **Repeat last hour:** if the latest count was 4,700, predict 4,700 again. **Typical weekday-hour:** use the median *training* count for the coming weekday and hour (for example, Tuesday at 5pm). This table is built from past data only. **MAE** is the average absolute error in vehicles per hour; lower is better.

In [ ]:
def mae(y, prediction):
    return mean_absolute_error(y, prediction)

typical = train.groupby(['next_weekday','next_hour']).next_volume.median()
def typical_hour_prediction(part):
    keys = list(zip(part.next_weekday,part.next_hour))
    return np.array([typical.loc[key] for key in keys])
print('Validation repeat-last-hour MAE:',round(mae(validation.next_volume,validation.latest_volume),1))
print('Validation typical-weekday-hour MAE:',round(mae(validation.next_volume,typical_hour_prediction(validation)),1))

## 6. Train a decision tree
A tree repeatedly asks simple questions such as `Is the next hour before 7am?` or `Was the last count above 3,000?`. At each split, training tries to reduce **squared error**. Each final leaf predicts the mean traffic count of training examples in that leaf. `max_depth` limits how many questions it asks, preventing a huge tree from memorising historical hours.

We choose the depth with the smallest **2017 validation MAE**. The 2018 test set is untouched until the next section.
**Theory to code:** For an example with true next-hour count `y` and predicted count `f(x)`, squared loss is `(y - f(x))²`. The tree chooses feature thresholds that reduce the sum of these losses in its training examples; a leaf returns their mean. `DecisionTreeRegressor` implements the tree, `.fit(train[features], train.next_volume)` learns its rules, and `.predict(...)` applies them. `min_samples_leaf=40` prevents decisions based on only a few historical hours; `max_depth` is selected on the 2017 validation set. Source: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html


**Design choices and alternatives:** A tree can learn nonlinear interactions between clock time and volume without requiring feature scaling, and its thresholds can be inspected. A linear model would be simpler but would need carefully designed time encodings/interactions to represent these patterns. An ensemble could reduce a single tree's instability, at the cost of more computation and less direct inspection; it has not been tested here, so no superiority is claimed. The two implemented baselines are empirical alternatives and establish whether extra complexity helps.

`min_samples_leaf=40` is a heuristic regularisation choice: averaging at least 40 examples avoids very small leaves, but could smooth rare events. It was fixed before evaluating 2018 and was not optimised. The depth grid is 3, 5, 7, 9 and 12; depth 12 is best within that grid, not proof of an optimum. A future sensitivity study could compare leaf sizes and deeper trees using historical validation only. `random_state=42` makes training repeatable. The default `criterion='squared_error'` and `splitter='best'` implement greedy threshold selection. There is no gradient-descent optimiser or learning rate: `.fit()` recursively chooses splits, and leaf means minimise squared error within each leaf. This is a computationally efficient heuristic, not a global search over all possible trees.


In [ ]:
depths = [3, 5, 7, 9, 12]
validation_scores = []
for depth in depths:
    candidate = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=40, random_state=42)
    candidate.fit(train[features], train.next_volume)
    prediction = candidate.predict(validation[features])
    validation_scores.append({'depth':depth,'validation_MAE':mae(validation.next_volume,prediction)})
score_table = pd.DataFrame(validation_scores)
print(score_table.round(1).to_string(index=False))
best_depth = int(score_table.loc[score_table.validation_MAE.idxmin(),'depth'])
model = DecisionTreeRegressor(max_depth=best_depth, min_samples_leaf=40, random_state=42)
model.fit(train[features],train.next_volume)
print('Chosen depth:',best_depth)
from sklearn.tree import export_text
print('First few learned rules (deeper branches omitted):')
print(export_text(model, feature_names=features, max_depth=2))


## 7. Evaluate once on 2018
Alongside MAE, RMSE makes large mistakes count more strongly. We also inspect rush hours separately: for road operations, a large miss at a busy time may matter more than the same error overnight. Rush hours here are a declared analysis choice, not a fact learned from the test set: 07:00-09:00 and 16:00-18:00 on weekdays.

**Metric interpretation:** MAE averages absolute count errors; it does not mean every hour is within that error or provide a percentage accuracy. RMSE is the square root of mean squared error and exposes the influence of large mistakes. The peak subset contains the six hourly bins 7, 8, 9, 16, 17 and 18 on weekdays; this is an operational proxy chosen for analysis, not a measured congestion label.


In [ ]:
y = test.next_volume.to_numpy()
baseline = test.latest_volume.to_numpy()
predicted = model.predict(test[features])
peak = (test.next_weekday.to_numpy() < 5) & (test.next_hour.isin([7,8,9,16,17,18]).to_numpy())

def metrics(actual, estimate, mask):
    return {'MAE': mean_absolute_error(actual[mask],estimate[mask]),
            'RMSE': np.sqrt(mean_squared_error(actual[mask],estimate[mask]))}
rows = []
for label, estimate in [('Repeat last hour',baseline),('Typical weekday-hour',typical_hour_prediction(test)),('Decision tree',predicted)]:
    for period, mask in [('all',np.ones(len(y),dtype=bool)),('weekday peaks',peak),('other hours',~peak)]:
        rows.append({'method':label,'period':period,'hours':int(mask.sum()),**metrics(y,estimate,mask)})
result_table = pd.DataFrame(rows)
print(result_table.round(1).to_string(index=False))

## 8. Look at actual mistakes
A table of errors answers more than an overall score. The plot shows one seven-day stretch of test data. The error table shows hours with the largest absolute mistakes, so we can explain what went wrong rather than claim the model is universally reliable.

**Limits of these diagnostics:** The plotted week is an illustration, while the tables assess the full retained test period. The worst-error rows reveal abrupt changes, but do not identify their causes. Feature importance describes training split contributions; correlated lag counts can share information, so low importance does not prove a feature is unnecessary and high importance does not establish causation.


In [ ]:
audit = test[['next_time','latest_volume','next_hour','next_weekday','next_volume']].copy()
audit['prediction'] = predicted
audit['typical_prediction'] = typical_hour_prediction(test)
audit['error_vehicles'] = audit.prediction - audit.next_volume
audit['absolute_error'] = audit.error_vehicles.abs()
print(audit.nlargest(8,'absolute_error').round(1).to_string(index=False))
week_start = audit.next_time.min()
week = audit[(audit.next_time >= week_start) & (audit.next_time < week_start + pd.Timedelta(days=7))]
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(week.next_time,week.next_volume,label='Observed',linewidth=1.5)
ax.plot(week.next_time,week.prediction,label='Predicted',linewidth=1.2)
ax.set(title='One week of held-out 2018 traffic',xlabel='Next hour',ylabel='Vehicles in hour')
ax.legend(); fig.autofmt_xdate(); plt.show()
print('Feature importance (relative split contribution):')
print(pd.Series(model.feature_importances_,index=features).sort_values(ascending=False).round(3).to_string())

## 9. Check generalisation and month-to-month stability
The training score is expected to be optimistic because the tree learned from those hours. Comparing train, validation, and test errors helps spot overfitting. We also report 2018 errors by calendar month: a single overall average can hide a weak month. Monthly variation is descriptive, not a claim that months are independent experiments.


**What the stability check establishes:** The train/validation/test MAEs of 164.9/181.3/172.4 do not suggest a large overfitting gap on these periods; they do not prove universal generalisation. The unweighted monthly mean MAE of 172.5 and standard deviation of 21.0 describe variability across nine months, whereas 172.4 is the hour-weighted overall test MAE. Months are not independent repeated experiments, so this standard deviation is not a confidence interval. The tree beats the typical weekday-hour baseline in each reported month. Stronger evidence for transfer across time would require rolling historical backtests; uncertainty estimates should respect temporal dependence, for example by resampling blocks of consecutive days rather than individual hours.


In [ ]:
for name, part in [('training',train),('validation',validation),('test',test)]:
    estimate = model.predict(part[features])
    print(name, 'MAE:',round(mae(part.next_volume,estimate),1))

month_rows = []
for month, group in audit.groupby(audit.next_time.dt.to_period('M')):
    month_rows.append({'month':str(month),'hours':len(group),
        'tree_MAE':(group.prediction-group.next_volume).abs().mean(),
        'typical_MAE':(group.typical_prediction-group.next_volume).abs().mean()})
monthly = pd.DataFrame(month_rows)
print(monthly.round(1).to_string(index=False))
print('Tree monthly MAE mean:',round(monthly.tree_MAE.mean(),1),
      'standard deviation:',round(monthly.tree_MAE.std(),1))


## 10. Compare intended and actual behavior
We expect the tree to follow recurring daily rises and falls while adapting to the latest count. The plotted week broadly shows this behavior, and the full-period results show an improvement over both baselines. However, weekday-peak MAE is 245.3, compared with 156.5 for other hours. Good average performance therefore does not imply equally good performance at busy times.

The largest test error occurs at 2018-05-21 13:00: volume falls from 4,810 to 2,040, while the tree predicts 4,831.5. At 14:00 volume rebounds to 4,224, while the prediction is 1,949.3. These paired errors show that recent counts can mislead the model during abrupt changes. An incident, closure or another event is a possible explanation, but the dataset does not establish the cause.

**Training loss versus practical objective:** Squared error penalises large numerical misses and makes a leaf mean easy to compute. MAE communicates average error in vehicles. Both are useful proxies for volume forecasting, but neither measures the consequences of a traffic-management decision. Squared error also penalises an overprediction and an equally sized underprediction symmetrically, although a specific operational use may have different costs.

A concrete hypothetical example illustrates the mismatch (these are not experimental results). Suppose two hours have true counts of 4,000 at a weekday peak and 1,000 off-peak. Predictor A outputs 3,400 and 1,000: MSE is 180,000 and MAE is 300. Predictor B outputs 3,900 and 1,700: MSE is 250,000 and MAE is 400. A wins on both ordinary metrics, but underpredicts the peak by 600 rather than 100 vehicles. If missing peak demand matters much more, B could better serve the practical objective. With an illustrative peak weight of 10, weighted MSE becomes about 327,273 for A and 53,636 for B. That weight is an example, not a validated business cost.

**Monitoring and specific improvements:** Keep reporting peak MAE and RMSE alongside overall MAE. A future operational evaluation could also monitor the fraction of peak hours underpredicted by more than a pre-agreed tolerance (for example 500 vehicles), and compare it with an acceptable service limit. Neither that tolerance nor a service limit has been validated here. If abrupt changes dominate errors, add incident/work-zone indicators genuinely available when forecasting. If peak underprediction dominates, test weighted squared-error training or an asymmetric loss with an appropriate regressor. A weighted tree could use training `sample_weight`; weights must be justified by the intended use and selected on validation data. These are proposed changes, not claimed improvements. Assess a revised model on a new untouched period because the 2018 errors have now informed these ideas. The current model was not retrained using those errors.


## 11. Make a prediction for a new hour
The deployment function takes only the three already-measured counts and the *upcoming* clock hour and weekday. It returns an estimated number of vehicles. Negative values would be physically impossible, so predictions are clipped at zero.

In [ ]:
def predict_next_hour(latest_volume, previous_1, previous_2, next_hour, next_weekday):
    new = pd.DataFrame([{'latest_volume':latest_volume,'previous_1':previous_1,
                         'previous_2':previous_2,'next_hour':next_hour,
                         'next_weekday':next_weekday}])
    return max(0.0,float(model.predict(new)[0]))

example = test.iloc[0]
print('Example prediction:',round(predict_next_hour(*[example[name] for name in features])),
      'vehicles; actual:',int(example.next_volume))

## 12. Conclusions from the completed run
The Colab run produced 35,843 valid forecasting examples from 40,575 unique measured hours. Training uses 20,696 examples, validation uses 8,650, and testing uses 6,497. The selected tree has depth 12 and a minimum leaf size of 40.

| Model | Test MAE | Test RMSE | Weekday-peak MAE |
|---|---:|---:|---:|
| Repeat latest count | 587.9 | 812.2 | 759.2 |
| Typical weekday-hour | 258.7 | 480.6 | 378.9 |
| Decision tree | 172.4 | 261.8 | 245.3 |

All errors are in vehicles per hour. The tree reduces overall MAE by approximately 33.4% against the stronger typical weekday-hour baseline, and by 70.7% against repeating the latest count. These are error reductions, not prediction accuracy percentages. It also improves peak-hour MAE and beats the typical baseline in each reported test month. The evidence supports the research question for this station and retained 2018 hours, while large individual errors prevent a claim of universal reliability.

**Limits:** This is one westbound measuring station. Counts alone do not establish travel speed, congestion, accidents or route alternatives. Missing hours are excluded, so scores do not assess operation with unavailable recent counts. Historical observations are from 2012–2018; present-day deployment requires current measurements and fresh validation. No operational intervention or benefit was evaluated.

**References:**
- Hogue, J. (2019). *Metro Interstate Traffic Volume* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C5X60B
- Federal Highway Administration. *Active Traffic Management Implementation and Operations Guide*, Chapter 4. https://ops.fhwa.dot.gov/publications/fhwahop19035/ch4.htm
- scikit-learn. *DecisionTreeRegressor* documentation. https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html

